### Container – Lösung Schritt für Schritt

*Jugendwettbewerb Informatik 2022, 3. Runde, Junioraufgabe 2 (= 41. Bundeswettbewerb Informatik, 1. Runde, Junioraufgabe 2)*

Mehrere Container werden per Kran auf ein Schiff verladen. Der schwerste Container soll zuerst auf das Schiff platziert
werden; das ist bei Seegang gut für den Schwerpunkt.

Jeder Container ist vorher mindestens einmal mit einem anderen Container auf der Container-Paar-Waage gewesen. Für jedes
gewogene Container-Paar ist danach nur bekannt, welcher der beiden Container der schwerere ist. Von den Containern haben
keine zwei das gleiche Gewicht.

Steht damit fest, welchen Container die Kranführerin zuerst verladen soll?

#### Junioraufgabe 2

Schreibe ein Programm, das die Kranführerin unterstützt.

Das Programm soll für die gewogenen Container-Paare einlesen, welcher Container jeweils der schwerere ist.

Anschließend soll das Programm ausgeben, ob daraus eindeutig der schwerste Container bestimmt werden kann. Falls ja, soll
das Programm außerdem ausgeben, welcher Container das ist.

Wende dein Programm mindestens auf alle Beispiele an, die du auf den BWINF-Webseiten findest, und dokumentiere die Ergebnisse.

#### Beispieldaten

Im Ordner `beispieldaten` stehen `container0.txt` bis `container4.txt`. Jede Zeile enthält ein gewogenes Paar `a b`: Der Container `a` ist
schwerer als der Container `b`. Die Container sind mit Zahlen bezeichnet. In `container0.txt` stehen zum Beispiel die Paare `4 1`, `3 2`,
`5 2` und `5 1`.

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Beziehungen als Pfeile (Graph):** Jedes Paar „a ist schwerer als b“ ist ein Pfeil von `a` nach `b`. Alle Container und Pfeile bilden einen
  **gerichteten Graphen**. Weil kein Container schwerer als er selbst sein kann, gibt es keine Kreise (azyklischer Graph).
- **Elimination:** Ein Container, der bei irgendeiner Wägung der leichtere war, kann nicht der schwerste sein. Alle anderen bleiben
  als **Kandidaten** übrig. Das ist eine einfache **Mengenoperation** (Menge aller Container minus Menge der leichteren).
- **Quelle im Graphen:** Der schwerste Container ist ein Knoten, in den kein Pfeil hineinführt. Gibt es genau einen solchen Knoten, ist er eindeutig.
- **Transitivität:** Ist `a` schwerer als `b` und `b` schwerer als `c`, so ist `a` auch schwerer als `c`. Das nutzt man für die Gegenprobe (Tiefensuche).

**Nützliche Funktionen und Datenstrukturen**
- **Menge (`set`)**: `set()` speichert jedes Element nur einmal, `menge.add(x)` fügt ein Element hinzu, `menge.discard(x)` entfernt es (ohne Fehler,
  wenn es nicht vorhanden ist), `len(menge)` zählt die Elemente, `sorted(menge)` macht eine sortierte Liste daraus.
- **Wörterbuch (`dict`)** für den Graphen: `leichter[a]` ist die Liste der Container, die direkt leichter als `a` sind (nur für die Gegenprobe).
- Eine Liste als Stapel für die Tiefensuche: `liste.append(x)` und `liste.pop()`

## Die Idee

Bei einer Wägung von `a` und `b` wissen wir: `a` ist schwerer, `b` kann deshalb nicht der schwerste Container sein.

1. Am Anfang kann jeder Container der schwerste sein: Alle sind **Kandidaten**.
2. Für jedes gewogene Paar `a b` streichen wir `b` aus den Kandidaten.
3. Am Ende bleiben die Container übrig, die bei **keiner** Wägung der leichtere waren.

Bleibt **genau ein** Kandidat übrig, muss er der schwerste sein: Jeder andere Container war schon einmal der leichtere, also kann keiner von ihnen
der schwerste sein.
Bleiben **mehrere** übrig, kann man nicht wissen, welcher von ihnen der schwerste ist, denn zwischen ihnen ist keine Wägung bekannt. Jeder von ihnen
war nie der leichtere, und sie sind alle unterschiedlich schwer.

## Schritt 1: Die Paare einlesen

Jede Zeile enthält zwei Zahlen. Wir speichern jedes Paar als kleine Liste `[schwerer, leichter]`. Alle Zahlen lesen wir mit `split()` und gehen sie
paarweise durch.

In [ ]:
def lies_paare(dateiname):
    with open(dateiname, encoding='utf-8') as f:
        zahlen = f.read().split()
    paare = []
    for i in range(0, len(zahlen), 2):          # in Zweierschritten: 0, 2, 4, ...
        schwerer = int(zahlen[i])
        leichter = int(zahlen[i + 1])
        paare.append([schwerer, leichter])
    return paare


paare = lies_paare('beispieldaten/container0.txt')
print(paare)

[[4, 1], [3, 2], [5, 2], [5, 1]]


## Schritt 2: Die Kandidaten bestimmen

Die Menge aller Container ergibt sich aus den Paaren: Jede Zahl, die irgendwo in einem Paar vorkommt, ist ein Container (nach der Aufgabe war jeder Container
bei mindestens einer Wägung dabei). Dann streichen wir für jedes Paar den leichteren Container aus den Kandidaten.
`menge.discard(x)` entfernt `x` und macht nichts, wenn `x` nicht (mehr) in der Menge ist.

In [ ]:
def finde_kandidaten(paare):
    kandidaten = set()
    for schwerer, leichter in paare:
        kandidaten.add(schwerer)
        kandidaten.add(leichter)
    for schwerer, leichter in paare:
        kandidaten.discard(leichter)            # der leichtere kann nicht der schwerste sein
    return kandidaten


print(finde_kandidaten(paare))

{3, 4, 5}


In `container0.txt` bleiben die Container 3, 4 und 5 übrig. Die Container 1 und 2 waren bei einer Wägung leichter, die anderen nicht.
Weil zwischen 3, 4 und 5 keine Wägung stattgefunden hat, kann man nicht sagen, welcher der schwerste ist.

## Schritt 3: Die Antwort ausgeben

Gibt es genau einen Kandidaten, ist er der schwerste Container. Sonst sagen wir, dass es nicht eindeutig ist, und nennen die Kandidaten.
`len(kandidaten)` ist die Anzahl der Elemente, `sorted(kandidaten)` macht eine sortierte Liste daraus (für die Ausgabe in einer festen Reihenfolge).

In [ ]:
def bearbeite(dateiname):
    paare = lies_paare(dateiname)
    kandidaten = finde_kandidaten(paare)
    print(f'{dateiname}: {len(paare)} Wägungen')
    if len(kandidaten) == 1:
        print(f'  Eindeutig: Der schwerste Container ist Container {sorted(kandidaten)[0]}.')
    else:
        print(f'  Nicht eindeutig: Als schwerster Container kommen in Frage: {sorted(kandidaten)}')


bearbeite('beispieldaten/container0.txt')

beispieldaten/container0.txt: 4 Wägungen
  Nicht eindeutig: Als schwerster Container kommen in Frage: [3, 4, 5]


## Schritt 4: Alle Beispiele

In [ ]:
for nr in range(5):
    bearbeite(f'beispieldaten/container{nr}.txt')

beispieldaten/container0.txt: 4 Wägungen
  Nicht eindeutig: Als schwerster Container kommen in Frage: [3, 4, 5]
beispieldaten/container1.txt: 6 Wägungen
  Eindeutig: Der schwerste Container ist Container 4.
beispieldaten/container2.txt: 15 Wägungen
  Nicht eindeutig: Als schwerster Container kommen in Frage: [1, 3]
beispieldaten/container3.txt: 20 Wägungen
  Nicht eindeutig: Als schwerster Container kommen in Frage: [5, 7]
beispieldaten/container4.txt: 150 Wägungen
  Eindeutig: Der schwerste Container ist Container 5.


Das stimmt mit den offiziellen Lösungshinweisen überein: `container1.txt` und `container4.txt` sind eindeutig (Container 4 beziehungsweise 5),
bei `container0.txt` ({3, 4, 5}), `container2.txt` ({1, 3}) und `container3.txt` ({5, 7}) bleiben mehrere Kandidaten.

Bei `container1.txt` wurde jeder Container mit jedem anderen verglichen. Dann steht die Reihenfolge ganz fest, und der schwerste ist eindeutig.

## Für Neugierige: Eine zweite Lösung zur Kontrolle

Eine andere Idee prüft die Behauptung „Container `c` ist schwerer als alle anderen“ ganz direkt. Aus den Paaren kennen wir nur die
**direkt** gewogenen Container. Wegen der Transitivität dürfen wir aber weiter schließen: Ist `a` schwerer als `b` und `b` schwerer als `c`, dann ist
`a` auch schwerer als `c`. Wir folgen den Pfeilen von `a` aus (Tiefensuche) und sammeln alle Container, die so erreichbar sind.
Der Container `a` ist sicher der schwerste, wenn alle anderen Container erreichbar sind.

Dazu bauen wir ein Wörterbuch `leichter` auf: Zu jedem schwereren Container die Liste der direkt leichteren. Beim Suchen führt die Liste `zu_pruefen`
alle Container, deren Nachfolger wir noch ansehen müssen. `pop()` nimmt den letzten Eintrag heraus (so arbeitet man einen Stapel ab).

In [ ]:
def erreichbare_container(paare, start):
    leichter = {}
    for schwerer, kleiner in paare:
        if schwerer not in leichter:
            leichter[schwerer] = []
        leichter[schwerer].append(kleiner)

    gesehen = set()
    zu_pruefen = [start]
    while len(zu_pruefen) > 0:
        container = zu_pruefen.pop()
        for naechster in leichter.get(container, []):
            if naechster not in gesehen:
                gesehen.add(naechster)
                zu_pruefen.append(naechster)
    return gesehen


def schwerste_pruefen(paare):
    alle = set()
    for schwerer, leichter in paare:
        alle.add(schwerer)
        alle.add(leichter)
    sicher = []
    for c in sorted(alle):
        if len(erreichbare_container(paare, c)) == len(alle) - 1:      # alle anderen sind sicher leichter
            sicher.append(c)
    return sicher


for nr in range(5):
    paare = lies_paare(f'beispieldaten/container{nr}.txt')
    kandidaten = finde_kandidaten(paare)
    print(f'container{nr}.txt: Kandidaten {sorted(kandidaten)}, sicher schwerster (Gegenprobe): {schwerste_pruefen(paare)}')

container0.txt: Kandidaten [3, 4, 5], sicher schwerster (Gegenprobe): []
container1.txt: Kandidaten [4], sicher schwerster (Gegenprobe): [4]
container2.txt: Kandidaten [1, 3], sicher schwerster (Gegenprobe): []
container3.txt: Kandidaten [5, 7], sicher schwerster (Gegenprobe): []
container4.txt: Kandidaten [5], sicher schwerster (Gegenprobe): [5]


Bei den eindeutigen Beispielen liefert die Gegenprobe genau den Container, bei den anderen die leere Liste. Das passt zur ersten Lösung.
Die Tiefensuche ist langsamer (sie muss für jeden Container einmal suchen), aber sie beweist die Aussage direkt.

**Hinweis:** Wir nehmen an, dass die Wägungen widerspruchsfrei sind: Es gibt nie „a schwerer als b“ und gleichzeitig „b schwerer als a“
(oder einen längeren Kreis). Sonst gäbe es überhaupt keinen schwersten Container. Ein solcher Widerspruch ließe sich daran erkennen, dass die
Menge der Kandidaten leer ist, obwohl es Container gibt.